In [1]:
# Importa as bibliotecas necessárias para validação cruzada, TF-IDF, regressão logística e registro dos resultados.

import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from registro_resultados import registrar_resultado

In [ ]:
# Define o arquivo, as colunas e os parâmetros utilizados na validação cruzada.

ARQUIVO_TREINO = "../data/train.xlsx"

COLUNA_TEXTO = "resp_text"
COLUNA_CLASSE = "clarity"

RANDOM_STATE = 42
N_SPLITS = 5

In [4]:
# Carrega o conjunto de dados, trata os textos e separa as variáveis preditoras e a variável alvo.

df = pd.read_excel(ARQUIVO_TREINO)

df[COLUNA_TEXTO] = (
    df[COLUNA_TEXTO]
    .fillna("")
    .astype(str)
)

print("Dimensões:", df.shape)

print("\nColunas:")
print(df.columns.tolist())

print("\nDistribuição das classes:")
print(df[COLUNA_CLASSE].value_counts())

X = df[COLUNA_TEXTO]
y = df[COLUNA_CLASSE]


Dimensões: (20092, 2)

Colunas:
['resp_text', 'clarity']

Distribuição das classes:
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64


In [5]:
# Cria uma validação cruzada estratificada de cinco folds com embaralhamento controlado por uma semente fixa.

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [6]:
# Cria um pipeline que executa TF-IDF e Regressão Logística dentro de cada fold da validação cruzada.

pipeline_baseline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

In [7]:
# Executa a validação cruzada calculando acurácia e F1 macro nos conjuntos de treino e validação.

scores = cross_validate(
    pipeline_baseline,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro"
    },
    n_jobs=-1,
    return_train_score=True
)

In [8]:
# Organiza os resultados de cada fold em uma tabela para facilitar a análise dos experimentos.

resultados_folds = pd.DataFrame({
    "fold": range(1, N_SPLITS + 1),
    "acuracia_treino": scores["train_accuracy"],
    "acuracia_validacao": scores["test_accuracy"],
    "f1_macro_treino": scores["train_f1_macro"],
    "f1_macro_validacao": scores["test_f1_macro"]
})

print("==============================")
print("VALIDAÇÃO CRUZADA - BASELINE")
print("==============================")

print(resultados_folds.to_string(index=False))

VALIDAÇÃO CRUZADA - BASELINE
 fold  acuracia_treino  acuracia_validacao  f1_macro_treino  f1_macro_validacao
    1         0.707771            0.452600         0.707092            0.452068
    2         0.710011            0.460314         0.709368            0.457225
    3         0.712766            0.459930         0.712272            0.458479
    4         0.709344            0.464410         0.708728            0.463269
    5         0.710526            0.448731         0.709718            0.447388


In [9]:
# Calcula as médias e os desvios das métricas obtidas nos cinco folds.

acuracia_media = float(
    scores["test_accuracy"].mean()
)

acuracia_desvio = float(
    scores["test_accuracy"].std()
)

f1_macro_medio = float(
    scores["test_f1_macro"].mean()
)

f1_macro_desvio = float(
    scores["test_f1_macro"].std()
)

acuracia_treino_media = float(
    scores["train_accuracy"].mean()
)

f1_treino_medio = float(
    scores["train_f1_macro"].mean()
)

print(f"Acurácia média:        {acuracia_media:.4f}")
print(f"Desvio padrão:         {acuracia_desvio:.4f}")
print(f"F1 macro médio:        {f1_macro_medio:.4f}")
print(f"Desvio F1 macro:       {f1_macro_desvio:.4f}")
print(f"Acurácia treino média: {acuracia_treino_media:.4f}")
print(f"F1 treino médio:       {f1_treino_medio:.4f}")

Acurácia média:        0.4572
Desvio padrão:         0.0057
F1 macro médio:        0.4557
Desvio F1 macro:       0.0055
Acurácia treino média: 0.7101
F1 treino médio:       0.7094


In [10]:
# Registra as métricas consolidadas da validação cruzada na tabela de experimentos.

resultado_baseline_cv = {
    "experimento": "5-Fold CV - Baseline",
    "representacao": "TF-IDF",
    "modelo": "Regressão Logística",
    "folds": int(N_SPLITS),
    "acuracia_media": acuracia_media,
    "desvio_acuracia": acuracia_desvio,
    "f1_macro_medio": f1_macro_medio,
    "desvio_f1_macro": f1_macro_desvio,
    "acuracia_treino_media": acuracia_treino_media,
    "f1_macro_treino_medio": f1_treino_medio
}

print("Resultado:")
print(resultado_baseline_cv)

registrar_resultado(resultado_baseline_cv)

Resultado:
{'experimento': '5-Fold CV - Baseline', 'representacao': 'TF-IDF', 'modelo': 'Regressão Logística', 'folds': 5, 'acuracia_media': 0.457196967961169, 'desvio_acuracia': 0.005692491726826011, 'f1_macro_medio': 0.4556857813123304, 'desvio_f1_macro': 0.00547020775310293, 'acuracia_treino_media': 0.7100835856833705, 'f1_macro_treino_medio': 0.7094355108401772}
Tabela consolidada salva em: resultados_experimentos.csv
